# 1. Data preparation: diabetes hospital readmission

## Setup

Use Python 3.12 and install `requirements.txt`. Run the cells in order from the project or `notebooks` folder.

In [1]:
from pathlib import Path
import hashlib
import json
import urllib.request
import zipfile

import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'config.json').exists():
    raise FileNotFoundError('Open the notebook from the GenoMining project folder.')

SEED = json.loads((ROOT / 'config.json').read_text())['seed']
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 12)
pd.set_option('display.precision', 4)

## Load the original dataset

Load the [UCI dataset](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008) and category mappings. Treat `?` as missing. Keep laboratory `None` as “not tested” and medication `No` as a valid category.

In [2]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip'
archive_path = RAW_DIR / 'uci-diabetes.zip'
required_files = ['diabetic_data.csv', 'IDS_mapping.csv']

if not all((RAW_DIR / name).exists() for name in required_files):
    if not archive_path.exists():
        urllib.request.urlretrieve(DATA_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        for name in required_files:
            member = next(item for item in archive.namelist() if item.endswith(name))
            (RAW_DIR / name).write_bytes(archive.read(member))

raw = pd.read_csv(
    RAW_DIR / 'diabetic_data.csv',
    keep_default_na=False, na_values=['?'], low_memory=False,
)
print(f'{len(raw):,} encounters, {raw.shape[1]} columns, {raw.patient_nbr.nunique():,} patients')
display(raw[['encounter_id', 'patient_nbr', 'age', 'time_in_hospital',
             'number_inpatient', 'A1Cresult', 'readmitted']].head())

101,766 encounters, 50 columns, 71,518 patients


,encounter_id,patient_nbr,age,time_in_hospital,number_inpatient,A1Cresult,readmitted
0,2278392,8222157,[0-10),1,0,None,NO
1,149190,55629189,[10-20),3,0,None,>30
2,64410,86047875,[20-30),2,1,None,NO
3,500364,82442376,[30-40),2,0,None,NO
4,16680,42519267,[40-50),1,0,None,NO


## Check the structure and feature types

There are 47 predictors: eight numerical and 39 categorical. The other columns are two identifiers and the outcome. Age bands and admission codes are categorical.

In [3]:
numeric_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses',
]
identifiers = ['encounter_id', 'patient_nbr']
candidate_features = raw.columns.drop(identifiers + ['readmitted']).tolist()
categorical_features = [name for name in candidate_features if name not in numeric_features]

structure = pd.Series({
    'Encounters': len(raw),
    'Unique patients': raw.patient_nbr.nunique(),
    'Candidate predictors': len(candidate_features),
    'Numerical predictors': len(numeric_features),
    'Categorical predictors': len(categorical_features),
})
display(structure.to_frame('Count'))
display(raw[numeric_features].describe().round(2))

,Count
Encounters,101766
Unique patients,71518
Candidate predictors,47
Numerical predictors,8
Categorical predictors,39


,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00
mean,4.40,43.10,1.34,16.02,0.37,0.20,0.64,7.42
std,2.99,19.67,1.71,8.13,1.27,0.93,1.26,1.93
min,1.00,1.00,0.00,1.00,0.00,0.00,0.00,1.00
25%,2.00,31.00,0.00,10.00,0.00,0.00,0.00,6.00
50%,4.00,44.00,1.00,15.00,0.00,0.00,0.00,8.00
75%,6.00,57.00,2.00,20.00,0.00,0.00,1.00,9.00
max,14.00,132.00,6.00,81.00,42.00,76.00,21.00,16.00


## Validate identifiers, ranges, and category meanings

Check duplicate encounters, missing IDs, target labels, and numerical ranges. Repeated patients are expected; duplicate encounter IDs are not.

In [4]:
checks = pd.Series({
    'Duplicate encounter IDs': raw.encounter_id.duplicated().sum(),
    'Exact duplicate rows': raw.duplicated().sum(),
    'Missing identifier cells': raw[identifiers].isna().sum().sum(),
    'Negative numeric cells': (raw[numeric_features] < 0).sum().sum(),
    'Noninteger numeric cells': (raw[numeric_features] % 1 != 0).sum().sum(),
    'Stays outside 1-14 days': (~raw.time_in_hospital.between(1, 14)).sum(),
})
display(checks.to_frame('Count'))
assert checks.eq(0).all()
assert raw.shape == (101766, 50)
assert raw.patient_nbr.nunique() == 71518
assert set(raw.readmitted.unique()) == {'<30', '>30', 'NO'}
assert raw.A1Cresult.eq('None').any()
assert raw.max_glu_serum.eq('None').any()

source_hash = hashlib.sha256((RAW_DIR / 'diabetic_data.csv').read_bytes()).hexdigest()
print('Source CSV SHA-256:', source_hash)

,Count
Duplicate encounter IDs,0
Exact duplicate rows,0
Missing identifier cells,0
Negative numeric cells,0
Noninteger numeric cells,0
Stays outside 1-14 days,0


Source CSV SHA-256: 0689e7ec031237dc63031b938805c48377748761a3b26acab621567afa24df97


### Some unknown values are stored as category codes

Some codes mean unavailable, unmapped, or invalid. Keep them as separate categories and count them separately from `?`.

In [5]:
unknown_codes = {
    'admission_type_id': [5, 6, 8],
    'admission_source_id': [9, 15, 17, 20, 21],
    'discharge_disposition_id': [18, 25, 26],
}
unknown_summary = []
for name, codes in unknown_codes.items():
    unknown_summary.append({
        'feature': name,
        'unknown_codes': ', '.join(map(str, codes)),
        'encounters': raw[name].isin(codes).sum(),
        'percent': raw[name].isin(codes).mean() * 100,
    })
display(pd.DataFrame(unknown_summary).round(2))
print('Unknown/Invalid gender entries:', raw.gender.eq('Unknown/Invalid').sum())

,feature,unknown_codes,encounters,percent
0,admission_type_id,"5, 6, 8",10396,10.22
1,admission_source_id,"9, 15, 17, 20, 21",7067,6.94
2,discharge_disposition_id,"18, 25, 26",4680,4.60


Unknown/Invalid gender entries: 3


## Define the eligible cohort and target

Exclude death and hospice discharges. Set `<30` to 1 and `>30` or `NO` to 0. `NO` means no recorded readmission in the source.

In [6]:
excluded_discharge_codes = [11, 13, 14, 19, 20, 21]
cohort = raw.loc[~raw.discharge_disposition_id.isin(excluded_discharge_codes)].copy()
cohort = cohort.reset_index(drop=True)
cohort['target'] = cohort.readmitted.eq('<30').astype('int8')

population = pd.DataFrame({
    'population': ['Original release', 'Eligible cohort'],
    'encounters': [len(raw), len(cohort)],
    'patients': [raw.patient_nbr.nunique(), cohort.patient_nbr.nunique()],
})
display(population)
display(cohort.groupby('readmitted', sort=False).agg(
    encounters=('target', 'size'), binary_target=('target', 'first'),
))
print(f'Excluded encounters: {len(raw) - len(cohort):,}')
print(f'Eligible early-readmission proportion: {cohort.target.mean():.2%}')

,population,encounters,patients
0,Original release,101766,71518
1,Eligible cohort,99343,69990


,encounters,binary_target
readmitted,,
NO,52527,0
>30,35502,0
<30,11314,1


Excluded encounters: 2,423
Eligible early-readmission proportion: 11.39%


## Separate patients before exploring or learning transformations

Use about 60% development, 20% calibration, and 20% test data. Keep each patient in one set. Use development data for EDA and preprocessing.

In [7]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
cohort['split'] = 'development'
cohort['partition_fold'] = -1

for fold, (_, indices) in enumerate(
    splitter.split(cohort, cohort.target, groups=cohort.patient_nbr)
):
    cohort.loc[indices, 'partition_fold'] = fold
    if fold == 3:
        cohort.loc[indices, 'split'] = 'calibration'
    elif fold == 4:
        cohort.loc[indices, 'split'] = 'test'

assert cohort.groupby('patient_nbr')['split'].nunique().eq(1).all()
assert cohort.partition_fold.ge(0).all()
split_summary = cohort.groupby('split').agg(
    encounters=('target', 'size'), patients=('patient_nbr', 'nunique'),
    early_readmissions=('target', 'sum'), positive_rate=('target', 'mean'),
)
display(split_summary)
print('Patient overlap between partitions: 0')

,encounters,patients,early_readmissions,positive_rate
split,,,,
calibration,19746,14025,2169,0.1098
development,59670,41960,6854,0.1149
test,19927,14005,2291,0.1150


Patient overlap between partitions: 0


## Save the handoff to notebook 2

Save the cohort and patient splits. Notebook 2 will remove IDs and outcomes from the predictors.

In [8]:
cohort.to_csv(PROCESSED_DIR / 'cohort.csv', index=False, na_rep='?')
cohort[identifiers + ['split', 'partition_fold']].to_csv(
    PROCESSED_DIR / 'splits.csv', index=False,
)
print('Saved data/processed/cohort.csv and data/processed/splits.csv')

Saved data/processed/cohort.csv and data/processed/splits.csv


## References

Clore et al. (2014). [UCI Diabetes 130-US Hospitals dataset](https://doi.org/10.24432/C5230J).